In [1]:
from pathlib import Path

import pandas as pd
import numpy as np


cwd = Path.cwd()

project_root = (
    cwd
    if (cwd / "data").exists()
    else cwd.parent
)

processed_dir = (
    project_root
    / "data"
    / "processed"
)

canonical_path = (
    processed_dir
    / "sa_budget_canonical.csv"
)

budget = pd.read_csv(
    canonical_path
)

print("Rows:", len(budget))
print("Columns:", len(budget.columns))

display(budget.head())

Rows: 155
Columns: 11


,source_document,source_page,source_url,program_name_raw,program_name_standardized,mapping_type,fiscal_year,metric_type,amount,raw_value,metric_context
0,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,ASC Book Fund,ASC Book Fund,same,2022-23,allocation,3000.00,"$3,000.00",NaN
1,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,"Equity, Diversity, Inclusion, and Community","Accessibility, Community, & Opportunity",renamed_or_reorganized,2022-23,allocation,11500.00,"$11,500.00",NaN
2,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Associated Student Council,Associated Student Council,same,2022-23,allocation,56953.00,"$56,953.00",NaN
3,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Bruce Mckenna Writing Center,Bruce McKenna Writing Center,spelling_variant,2022-23,allocation,56039.01,"$56,039.01",NaN
4,2023-24_sa_fee_memo.pdf,3,https://studentleadership.seattlecentral.edu/s...,Cultural Programming & Development (CAB),Cultural Programming & Development (CAB),same,2022-23,allocation,116767.00,"$116,767.00",NaN


In [2]:
allocation_totals = (
    budget[
        budget["metric_type"]
        == "allocation"
    ]
    .groupby(
        "fiscal_year",
        as_index=False
    )
    ["amount"]
    .sum()
)

display(allocation_totals)

,fiscal_year,amount
0,2022-23,1500000.00
1,2023-24,1513171.18
2,2024-25,1733008.51
3,2025-26,1801888.48


In [3]:
allocation_totals[
    "allocation_change"
] = (
    allocation_totals["amount"]
    .diff()
)

allocation_totals[
    "allocation_change_pct"
] = (
    allocation_totals["amount"]
    .pct_change()
    * 100
)

display(allocation_totals)

,fiscal_year,amount,allocation_change,allocation_change_pct
0,2022-23,1500000.00,NaN,NaN
1,2023-24,1513171.18,13171.18,0.878079
2,2024-25,1733008.51,219837.33,14.528253
3,2025-26,1801888.48,68879.97,3.974589


In [4]:
allocation_summary = (
    allocation_totals.copy()
)

allocation_summary["amount_display"] = (
    allocation_summary["amount"]
    .map(lambda x: f"${x:,.2f}")
)

allocation_summary["change_display"] = (
    allocation_summary["allocation_change"]
    .map(
        lambda x:
        ""
        if pd.isna(x)
        else f"${x:,.2f}"
    )
)

allocation_summary["change_pct_display"] = (
    allocation_summary[
        "allocation_change_pct"
    ]
    .map(
        lambda x:
        ""
        if pd.isna(x)
        else f"{x:.1f}%"
    )
)

display(
    allocation_summary[
        [
            "fiscal_year",
            "amount_display",
            "change_display",
            "change_pct_display",
        ]
    ]
)

,fiscal_year,amount_display,change_display,change_pct_display
0,2022-23,"$1,500,000.00",,
1,2023-24,"$1,513,171.18","$13,171.18",0.9%
2,2024-25,"$1,733,008.51","$219,837.33",14.5%
3,2025-26,"$1,801,888.48","$68,879.97",4.0%


In [5]:
first_year_amount = (
    allocation_totals.iloc[0]["amount"]
)

latest_year_amount = (
    allocation_totals.iloc[-1]["amount"]
)

overall_change = (
    latest_year_amount
    - first_year_amount
)

overall_change_pct = (
    overall_change
    / first_year_amount
    * 100
)

print(
    "2022-23 allocation:",
    f"${first_year_amount:,.2f}"
)

print(
    "2025-26 allocation:",
    f"${latest_year_amount:,.2f}"
)

print(
    "Total increase:",
    f"${overall_change:,.2f}"
)

print(
    "Overall increase:",
    f"{overall_change_pct:.1f}%"
)

2022-23 allocation: $1,500,000.00
2025-26 allocation: $1,801,888.48
Total increase: $301,888.48
Overall increase: 20.1%


In [6]:
allocations = budget[
    (
        budget["metric_type"] == "allocation"
    )
    &
    (
        budget["amount"].notna()
    )
].copy()

print("Allocation rows:", len(allocations))

Allocation rows: 89


In [7]:
allocations["allocation_rank"] = (
    allocations
    .groupby("fiscal_year")["amount"]
    .rank(
        method="min",
        ascending=False,
    )
)

allocations["budget_share_pct"] = (
    allocations["amount"]
    /
    allocations.groupby(
        "fiscal_year"
    )["amount"].transform("sum")
    * 100
)

In [8]:
top_10_by_year = (
    allocations[
        allocations["allocation_rank"] <= 10
    ]
    [
        [
            "fiscal_year",
            "allocation_rank",
            "program_name_standardized",
            "amount",
            "budget_share_pct",
        ]
    ]
    .sort_values(
        [
            "fiscal_year",
            "allocation_rank",
        ]
    )
    .reset_index(drop=True)
)

display(top_10_by_year)

,fiscal_year,allocation_rank,program_name_standardized,amount,budget_share_pct
0,2022-23,1.0,Learning Support Network,419159.00,27.943933
1,2022-23,2.0,Information Central,157950.76,10.530051
2,2022-23,3.0,Student Resource Support,136607.00,9.107133
3,2022-23,4.0,Student Leadership Program,131195.78,8.746385
4,2022-23,5.0,Student Organization Hub,122220.00,8.148000
5,2022-23,6.0,Cultural Programming & Development (CAB),116767.00,7.784467
6,2022-23,7.0,Office Management,71667.33,4.777822
7,2022-23,8.0,M. Rosetta Hunter Art Gallery,57513.12,3.834208
8,2022-23,9.0,Associated Student Council,56953.00,3.796867
9,2022-23,10.0,Bruce McKenna Writing Center,56039.01,3.735934


In [9]:
top_10_display = top_10_by_year.copy()

top_10_display["allocation"] = (
    top_10_display["amount"]
    .map(lambda x: f"${x:,.2f}")
)

top_10_display["budget_share"] = (
    top_10_display["budget_share_pct"]
    .map(lambda x: f"{x:.1f}%")
)

display(
    top_10_display[
        [
            "fiscal_year",
            "allocation_rank",
            "program_name_standardized",
            "allocation",
            "budget_share",
        ]
    ]
)

,fiscal_year,allocation_rank,program_name_standardized,allocation,budget_share
0,2022-23,1.0,Learning Support Network,"$419,159.00",27.9%
1,2022-23,2.0,Information Central,"$157,950.76",10.5%
2,2022-23,3.0,Student Resource Support,"$136,607.00",9.1%
3,2022-23,4.0,Student Leadership Program,"$131,195.78",8.7%
4,2022-23,5.0,Student Organization Hub,"$122,220.00",8.1%
5,2022-23,6.0,Cultural Programming & Development (CAB),"$116,767.00",7.8%
6,2022-23,7.0,Office Management,"$71,667.33",4.8%
7,2022-23,8.0,M. Rosetta Hunter Art Gallery,"$57,513.12",3.8%
8,2022-23,9.0,Associated Student Council,"$56,953.00",3.8%
9,2022-23,10.0,Bruce McKenna Writing Center,"$56,039.01",3.7%


In [10]:
latest_top_10 = (
    top_10_display[
        top_10_display["fiscal_year"]
        == "2025-26"
    ]
)

display(latest_top_10)

,fiscal_year,allocation_rank,program_name_standardized,amount,budget_share_pct,allocation,budget_share
30,2025-26,1.0,Learning Support Network,477762.58,26.514548,"$477,762.58",26.5%
31,2025-26,2.0,Student Leadership Program,186665.23,10.359422,"$186,665.23",10.4%
32,2025-26,3.0,Information Central,186282.00,10.338154,"$186,282.00",10.3%
33,2025-26,4.0,Cultural Programming & Development (CAB),148650.92,8.249729,"$148,650.92",8.2%
34,2025-26,5.0,Student Organization Hub,134474.20,7.462959,"$134,474.20",7.5%
35,2025-26,6.0,Associated Student Council,96748.00,5.369256,"$96,748.00",5.4%
36,2025-26,7.0,Student Support Program Supervisor,96277.04,5.343119,"$96,277.04",5.3%
37,2025-26,8.0,Office Management,89492.00,4.966567,"$89,492.00",5.0%
38,2025-26,9.0,Seattle Collegian,83577.00,4.638300,"$83,577.00",4.6%
39,2025-26,10.0,M. Rosetta Hunter Art Gallery,77930.27,4.324922,"$77,930.27",4.3%


In [11]:
allocation_pivot = (
    allocations
    .pivot_table(
        index="program_name_standardized",
        columns="fiscal_year",
        values="amount",
        aggfunc="sum",
    )
    .reset_index()
)

display(allocation_pivot)

fiscal_year,program_name_standardized,2022-23,2023-24,2024-25,2025-26
0,AANAPISI,NaN,0.00,0.00,36630.28
1,ASC Book Fund,3000.00,2000.00,2000.00,0.00
2,"Accessibility, Community, & Opportunity",11500.00,9000.00,9076.80,10700.00
3,Associated Student Council,56953.00,64539.00,93159.58,96748.00
4,Bruce McKenna Writing Center,56039.01,NaN,NaN,NaN
5,Child Assist Program,NaN,20000.00,20000.00,NaN
6,Cultural Programming & Development (CAB),116767.00,116767.00,128841.99,148650.92
7,Emergency Fund,25000.00,25000.00,25000.00,50000.00
8,First Year Experience Peer Mentors,NaN,0.00,0.00,20314.25
9,Food and Resource Pantry,NaN,0.00,0.00,30000.00


In [12]:
allocation_pivot[
    "change_2022_23_to_2025_26"
] = (
    allocation_pivot["2025-26"]
    - allocation_pivot["2022-23"]
)

allocation_pivot[
    "change_pct_2022_23_to_2025_26"
] = (
    allocation_pivot[
        "change_2022_23_to_2025_26"
    ]
    /
    allocation_pivot["2022-23"]
    * 100
)

In [13]:
comparable_programs = (
    allocation_pivot[
        allocation_pivot["2022-23"].notna()
        &
        allocation_pivot["2025-26"].notna()
    ]
    .sort_values(
        "change_2022_23_to_2025_26",
        ascending=False,
    )
)

display(
    comparable_programs[
        [
            "program_name_standardized",
            "2022-23",
            "2025-26",
            "change_2022_23_to_2025_26",
            "change_pct_2022_23_to_2025_26",
        ]
    ]
)

fiscal_year,program_name_standardized,2022-23,2025-26,change_2022_23_to_2025_26,change_pct_2022_23_to_2025_26
12,Learning Support Network,419159.00,477762.58,58603.58,13.981229
20,Student Leadership Program,131195.78,186665.23,55469.45,42.279904
3,Associated Student Council,56953.00,96748.00,39795.00,69.873404
6,Cultural Programming & Development (CAB),116767.00,148650.92,31883.92,27.305591
17,Seattle Collegian,52073.00,83577.00,31504.00,60.499683
10,Information Central,157950.76,186282.00,28331.24,17.936754
7,Emergency Fund,25000.00,50000.00,25000.00,100.000000
13,M. Rosetta Hunter Art Gallery,57513.12,77930.27,20417.15,35.499987
14,Office Management,71667.33,89492.00,17824.67,24.871402
21,Student Organization Hub,122220.00,134474.20,12254.20,10.026346


In [14]:
display(
    comparable_programs[
        [
            "program_name_standardized",
            "2022-23",
            "2025-26",
            "change_2022_23_to_2025_26",
        ]
    ].head(10)
)

fiscal_year,program_name_standardized,2022-23,2025-26,change_2022_23_to_2025_26
12,Learning Support Network,419159.00,477762.58,58603.58
20,Student Leadership Program,131195.78,186665.23,55469.45
3,Associated Student Council,56953.00,96748.00,39795.00
6,Cultural Programming & Development (CAB),116767.00,148650.92,31883.92
17,Seattle Collegian,52073.00,83577.00,31504.00
10,Information Central,157950.76,186282.00,28331.24
7,Emergency Fund,25000.00,50000.00,25000.00
13,M. Rosetta Hunter Art Gallery,57513.12,77930.27,20417.15
14,Office Management,71667.33,89492.00,17824.67
21,Student Organization Hub,122220.00,134474.20,12254.20


In [15]:
allocation_pivot.columns.name = None

In [16]:
decreasing_programs = (
    comparable_programs[
        comparable_programs[
            "change_2022_23_to_2025_26"
        ] < 0
    ]
    .sort_values(
        "change_2022_23_to_2025_26"
    )
)

display(
    decreasing_programs[
        [
            "program_name_standardized",
            "2022-23",
            "2025-26",
            "change_2022_23_to_2025_26",
        ]
    ]
)

fiscal_year,program_name_standardized,2022-23,2025-26,change_2022_23_to_2025_26
26,Wood Technology Center,12000.0,4000.0,-8000.0
1,ASC Book Fund,3000.0,0.0,-3000.0
2,"Accessibility, Community, & Opportunity",11500.0,10700.0,-800.0


In [17]:
newer_programs = (
    allocation_pivot[
        allocation_pivot["2022-23"].isna()
        &
        allocation_pivot["2025-26"].notna()
        &
        (allocation_pivot["2025-26"] > 0)
    ]
    .sort_values(
        "2025-26",
        ascending=False,
    )
)

display(
    newer_programs[
        [
            "program_name_standardized",
            "2025-26",
        ]
    ]
)

,program_name_standardized,2025-26
24,Student Support Program Supervisor,96277.04
0,AANAPISI,36630.28
9,Food and Resource Pantry,30000.00
8,First Year Experience Peer Mentors,20314.25
18,Services & Activities Fees Committee,7867.10
25,Umoja Scholars Program,7500.00


In [18]:
historical_programs = (
    allocation_pivot[
        allocation_pivot["2022-23"].notna()
        &
        (allocation_pivot["2022-23"] > 0)
        &
        (
            allocation_pivot["2025-26"].isna()
            |
            (allocation_pivot["2025-26"] == 0)
        )
    ]
    .sort_values(
        "2022-23",
        ascending=False,
    )
)

display(
    historical_programs[
        [
            "program_name_standardized",
            "2022-23",
            "2025-26",
        ]
    ]
)

,program_name_standardized,2022-23,2025-26
22,Student Resource Support,136607.00,NaN
4,Bruce McKenna Writing Center,56039.01,NaN
15,Parent Support Network,25000.00,NaN
1,ASC Book Fund,3000.00,0.0


In [19]:
requests = budget[
    (
        budget["metric_type"] == "request"
    )
    &
    (
        budget["amount"].notna()
    )
].copy()

request_pivot = (
    requests
    .pivot_table(
        index="program_name_standardized",
        columns="fiscal_year",
        values="amount",
        aggfunc="sum",
    )
)

allocation_request_compare = (
    budget[
        budget["fiscal_year"].isin(
            [
                "2023-24",
                "2024-25",
                "2025-26",
            ]
        )
    ]
    .pivot_table(
        index=[
            "program_name_standardized",
            "fiscal_year",
        ],
        columns="metric_type",
        values="amount",
        aggfunc="sum",
    )
    .reset_index()
)

allocation_request_compare.columns.name = None

display(
    allocation_request_compare.head(20)
)

,program_name_standardized,fiscal_year,allocation,request
0,AANAPISI,2023-24,0.00,NaN
1,AANAPISI,2024-25,0.00,NaN
2,AANAPISI,2025-26,36630.28,44220.15
3,ASC Book Fund,2023-24,2000.00,3000.00
4,ASC Book Fund,2024-25,2000.00,0.00
5,ASC Book Fund,2025-26,0.00,3000.00
6,"Accessibility, Community, & Opportunity",2023-24,9000.00,11500.00
7,"Accessibility, Community, & Opportunity",2024-25,9076.80,0.00
8,"Accessibility, Community, & Opportunity",2025-26,10700.00,11700.00
9,Associated Student Council,2023-24,64539.00,64539.00


In [20]:
allocation_request_compare[
    "allocation_minus_request"
] = (
    allocation_request_compare["allocation"]
    -
    allocation_request_compare["request"]
)

allocation_request_compare[
    "funded_pct"
] = (
    allocation_request_compare["allocation"]
    /
    allocation_request_compare["request"]
    * 100
)

In [21]:
full_request_years = (
    allocation_request_compare[
        allocation_request_compare[
            "fiscal_year"
        ].isin(
            [
                "2023-24",
                "2025-26",
            ]
        )
        &
        allocation_request_compare[
            "request"
        ].notna()
        &
        allocation_request_compare[
            "allocation"
        ].notna()
    ]
    .copy()
)

In [22]:
largest_request_reductions = (
    full_request_years
    .sort_values(
        "allocation_minus_request"
    )
)

display(
    largest_request_reductions[
        [
            "fiscal_year",
            "program_name_standardized",
            "request",
            "allocation",
            "allocation_minus_request",
            "funded_pct",
        ]
    ].head(15)
)

,fiscal_year,program_name_standardized,request,allocation,allocation_minus_request,funded_pct
34,2023-24,Learning Support Network,540680.00,470430.25,-70249.75,87.007148
30,2025-26,Information Central,238402.00,186282.00,-52120.00,78.137767
24,2025-26,First Year Experience Peer Mentors,57000.00,20314.25,-36685.75,35.639035
58,2025-26,Student Leadership Program,220539.00,186665.23,-33873.77,84.640463
49,2025-26,Seattle Collegian,111880.40,83577.00,-28303.40,74.702093
36,2025-26,Learning Support Network,503767.04,477762.58,-26004.46,94.837999
19,2023-24,Emergency Fund,50000.00,25000.00,-25000.00,50.000000
71,2025-26,Umoja Scholars Program,29500.00,7500.00,-22000.00,25.423729
11,2025-26,Associated Student Council,116663.00,96748.00,-19915.00,82.929463
39,2025-26,M. Rosetta Hunter Art Gallery,94729.09,77930.27,-16798.82,82.266461


In [23]:
request_year_summary = (
    full_request_years
    .groupby(
        "fiscal_year",
        as_index=False,
    )
    .agg(
        total_request=("request", "sum"),
        total_allocation=("allocation", "sum"),
    )
)

request_year_summary[
    "difference"
] = (
    request_year_summary["total_allocation"]
    -
    request_year_summary["total_request"]
)

request_year_summary[
    "funded_pct"
] = (
    request_year_summary["total_allocation"]
    /
    request_year_summary["total_request"]
    * 100
)

display(request_year_summary)

,fiscal_year,total_request,total_allocation,difference,funded_pct
0,2023-24,1608326.63,1493171.18,-115155.45,92.840046
1,2025-26,2046910.19,1801888.48,-245021.71,88.029680


In [24]:
def classify_funding_outcome(row):
    request = row["request"]
    allocation = row["allocation"]

    if pd.isna(request) or pd.isna(allocation):
        return "Missing comparison"

    if request == 0:
        if allocation == 0:
            return "Zero request / zero allocation"
        return "Allocation with zero request"

    difference = allocation - request

    if abs(difference) < 0.01:
        return "Fully funded"

    if difference > 0:
        return "Above request"

    if allocation == 0:
        return "No allocation"

    return "Partially funded"


full_request_years[
    "funding_outcome"
] = full_request_years.apply(
    classify_funding_outcome,
    axis=1,
)

In [25]:
outcome_summary = (
    full_request_years
    .groupby(
        [
            "fiscal_year",
            "funding_outcome",
        ]
    )
    .size()
    .reset_index(name="programs")
)

display(outcome_summary)

,fiscal_year,funding_outcome,programs
0,2023-24,Above request,3
1,2023-24,Allocation with zero request,1
2,2023-24,Fully funded,6
3,2023-24,Partially funded,7
4,2025-26,Above request,4
5,2025-26,Fully funded,1
6,2025-26,No allocation,1
7,2025-26,Partially funded,15
8,2025-26,Zero request / zero allocation,3


In [26]:
above_request = (
    full_request_years[
        full_request_years[
            "allocation_minus_request"
        ] > 0.01
    ]
    .sort_values(
        "allocation_minus_request",
        ascending=False,
    )
)

display(
    above_request[
        [
            "fiscal_year",
            "program_name_standardized",
            "request",
            "allocation",
            "allocation_minus_request",
        ]
    ]
)

,fiscal_year,program_name_standardized,request,allocation,allocation_minus_request
27,2025-26,Food and Resource Pantry,15000.00,30000.00,15000.00
42,2025-26,Office Management,78093.62,89492.00,11398.38
31,2023-24,Leadership & Orientation Training,8500.00,16384.00,7884.00
68,2025-26,Student Support Program Supervisor,92119.04,96277.04,4158.00
72,2023-24,Wood Technology Center,0.00,4000.00,4000.00
62,2023-24,Student Resource Support,136000.00,139203.30,3203.30
50,2023-24,Services & Activities Fees Committee,5416.00,7500.00,2084.00
52,2025-26,Services & Activities Fees Committee,7600.00,7867.10,267.10


In [27]:
reported_path = (
    processed_dir
    / "sa_budget_reported_long.csv"
)

reported = pd.read_csv(
    reported_path
)

In [28]:
decision_year_sources = {
    "2023-24": "2023-24_sa_fee_memo.pdf",
    "2024-25": "2024-25_sa_fee_memo.pdf",
    "2025-26": "2025-26_sa_budget_summary.pdf",
}

In [29]:
decision_rows = []

for fiscal_year, source_document in decision_year_sources.items():

    part = reported[
        (
            reported["fiscal_year"] == fiscal_year
        )
        &
        (
            reported["source_document"] == source_document
        )
        &
        (
            reported["metric_type"].isin(
                ["request", "allocation"]
            )
        )
    ].copy()

    decision_rows.append(part)


decision_budget = pd.concat(
    decision_rows,
    ignore_index=True
)

In [30]:
decision_compare = (
    decision_budget
    .pivot_table(
        index=[
            "program_name_standardized",
            "fiscal_year",
        ],
        columns="metric_type",
        values="amount",
        aggfunc="sum",
    )
    .reset_index()
)

decision_compare.columns.name = None

In [31]:
decision_compare[
    "allocation_minus_request"
] = (
    decision_compare["allocation"]
    -
    decision_compare["request"]
)

decision_compare[
    "funded_pct"
] = (
    decision_compare["allocation"]
    /
    decision_compare["request"]
    * 100
)

In [32]:
decision_year_summary = (
    decision_compare
    .groupby(
        "fiscal_year",
        as_index=False,
    )
    .agg(
        total_request=("request", "sum"),
        total_allocation=("allocation", "sum"),
    )
)

decision_year_summary["difference"] = (
    decision_year_summary["total_allocation"]
    -
    decision_year_summary["total_request"]
)

decision_year_summary["funded_pct"] = (
    decision_year_summary["total_allocation"]
    /
    decision_year_summary["total_request"]
    * 100
)

display(decision_year_summary)

,fiscal_year,total_request,total_allocation,difference,funded_pct
0,2023-24,1633326.63,1500000.00,-133326.63,91.837112
1,2024-25,1502598.56,1733008.51,230409.95,115.334099
2,2025-26,2046910.19,1801888.48,-245021.71,88.029680


In [33]:
def classify_decision_outcome(row):
    request = row["request"]
    allocation = row["allocation"]

    if pd.isna(request) or pd.isna(allocation):
        return "Missing comparison"

    if request == 0:
        if allocation == 0:
            return "Zero request / zero allocation"
        return "Allocation with zero request"

    difference = allocation - request

    if abs(difference) < 0.01:
        return "Fully funded"

    if difference > 0:
        return "Above request"

    if allocation == 0:
        return "No allocation"

    return "Partially funded"


decision_compare["funding_outcome"] = (
    decision_compare.apply(
        classify_decision_outcome,
        axis=1,
    )
)

In [34]:
decision_outcome_summary = (
    decision_compare
    .groupby(
        [
            "fiscal_year",
            "funding_outcome",
        ]
    )
    .size()
    .reset_index(name="programs")
)

display(decision_outcome_summary)

,fiscal_year,funding_outcome,programs
0,2023-24,Allocation with zero request,1
1,2023-24,Fully funded,10
2,2023-24,No allocation,1
3,2023-24,Partially funded,7
4,2023-24,Zero request / zero allocation,1
5,2024-25,Above request,3
6,2024-25,Allocation with zero request,8
7,2024-25,Fully funded,5
8,2024-25,Partially funded,3
9,2025-26,Above request,4


In [35]:
decision_above_request = (
    decision_compare[
        decision_compare[
            "allocation_minus_request"
        ] > 0.01
    ]
    .sort_values(
        [
            "fiscal_year",
            "allocation_minus_request",
        ],
        ascending=[True, False],
    )
)

display(
    decision_above_request[
        [
            "fiscal_year",
            "program_name_standardized",
            "request",
            "allocation",
            "allocation_minus_request",
        ]
    ]
)

,fiscal_year,program_name_standardized,request,allocation,allocation_minus_request
60,2023-24,Wood Technology Center,0.00,4000.00,4000.00
55,2024-25,Student Resource Support,0.00,139203.30,139203.30
46,2024-25,Student Engagement,0.00,45714.70,45714.70
38,2024-25,Phi Theta Kappa,0.00,36047.71,36047.71
17,2024-25,Emergency Fund,0.00,25000.00,25000.00
11,2024-25,Child Assist Program,0.00,20000.00,20000.00
25,2024-25,Leadership & Orientation Training,0.00,16384.00,16384.00
5,2024-25,"Accessibility, Community, & Opportunity",0.00,9076.80,9076.80
14,2024-25,Cultural Programming & Development (CAB),122613.11,128841.99,6228.88
52,2024-25,Student Organization Hub,123000.00,127811.35,4811.35


In [36]:
decision_above_request = (
    decision_compare[
        (decision_compare["request"] > 0)
        &
        (
            decision_compare[
                "allocation_minus_request"
            ] > 0.01
        )
    ]
    .sort_values(
        [
            "fiscal_year",
            "allocation_minus_request",
        ],
        ascending=[True, False],
    )
)

display(
    decision_above_request[
        [
            "fiscal_year",
            "program_name_standardized",
            "request",
            "allocation",
            "allocation_minus_request",
        ]
    ]
)

,fiscal_year,program_name_standardized,request,allocation,allocation_minus_request
14,2024-25,Cultural Programming & Development (CAB),122613.11,128841.99,6228.88
52,2024-25,Student Organization Hub,123000.00,127811.35,4811.35
22,2024-25,Information Central,191489.40,195205.32,3715.92
20,2025-26,Food and Resource Pantry,15000.00,30000.00,15000.00
35,2025-26,Office Management,78093.62,89492.00,11398.38
58,2025-26,Student Support Program Supervisor,92119.04,96277.04,4158.00
45,2025-26,Services & Activities Fees Committee,7600.00,7867.10,267.10


In [37]:
allocation_with_zero_request = (
    decision_compare[
        (decision_compare["request"] == 0)
        &
        (decision_compare["allocation"] > 0)
    ]
    .sort_values(
        [
            "fiscal_year",
            "allocation",
        ],
        ascending=[True, False],
    )
)

display(
    allocation_with_zero_request[
        [
            "fiscal_year",
            "program_name_standardized",
            "request",
            "allocation",
        ]
    ]
)

,fiscal_year,program_name_standardized,request,allocation
60,2023-24,Wood Technology Center,0.0,4000.00
55,2024-25,Student Resource Support,0.0,139203.30
46,2024-25,Student Engagement,0.0,45714.70
38,2024-25,Phi Theta Kappa,0.0,36047.71
17,2024-25,Emergency Fund,0.0,25000.00
11,2024-25,Child Assist Program,0.0,20000.00
25,2024-25,Leadership & Orientation Training,0.0,16384.00
5,2024-25,"Accessibility, Community, & Opportunity",0.0,9076.80
2,2024-25,ASC Book Fund,0.0,2000.00


In [38]:
concentration_rows = []

for year, group in allocations.groupby(
    "fiscal_year"
):
    group = group.sort_values(
        "amount",
        ascending=False,
    ).copy()

    total = group["amount"].sum()

    top_1_share = (
        group.head(1)["amount"].sum()
        / total
        * 100
    )

    top_3_share = (
        group.head(3)["amount"].sum()
        / total
        * 100
    )

    top_5_share = (
        group.head(5)["amount"].sum()
        / total
        * 100
    )

    top_10_share = (
        group.head(10)["amount"].sum()
        / total
        * 100
    )

    concentration_rows.append(
        {
            "fiscal_year": year,
            "top_1_share_pct": top_1_share,
            "top_3_share_pct": top_3_share,
            "top_5_share_pct": top_5_share,
            "top_10_share_pct": top_10_share,
        }
    )


budget_concentration = pd.DataFrame(
    concentration_rows
)

display(budget_concentration)

,fiscal_year,top_1_share_pct,top_3_share_pct,top_5_share_pct,top_10_share_pct
0,2022-23,27.943933,47.581117,64.475503,88.404800
1,2023-24,31.089031,50.844561,67.974542,92.143387
2,2024-25,27.316188,48.356066,63.823116,89.477574
3,2025-26,26.514548,47.212123,62.924812,87.566975


In [39]:
concentration_display = (
    budget_concentration.copy()
)

for column in [
    "top_1_share_pct",
    "top_3_share_pct",
    "top_5_share_pct",
    "top_10_share_pct",
]:
    concentration_display[column] = (
        concentration_display[column]
        .map(lambda x: f"{x:.1f}%")
    )

display(concentration_display)

,fiscal_year,top_1_share_pct,top_3_share_pct,top_5_share_pct,top_10_share_pct
0,2022-23,27.9%,47.6%,64.5%,88.4%
1,2023-24,31.1%,50.8%,68.0%,92.1%
2,2024-25,27.3%,48.4%,63.8%,89.5%
3,2025-26,26.5%,47.2%,62.9%,87.6%


In [40]:
programs_to_80 = []

for year, group in allocations.groupby(
    "fiscal_year"
):
    group = group.sort_values(
        "amount",
        ascending=False,
    ).copy()

    total = group["amount"].sum()

    group["cumulative_share"] = (
        group["amount"].cumsum()
        / total
        * 100
    )

    count_to_80 = (
        group["cumulative_share"]
        .le(80)
        .sum()
    )

    # Include the program that crosses 80%
    if (
        len(group) > count_to_80
        and count_to_80 < len(group)
    ):
        count_to_80 += 1

    programs_to_80.append(
        {
            "fiscal_year": year,
            "programs_needed_for_80_pct":
                count_to_80,
        }
    )


programs_to_80 = pd.DataFrame(
    programs_to_80
)

display(programs_to_80)

,fiscal_year,programs_needed_for_80_pct
0,2022-23,8
1,2023-24,7
2,2024-25,8
3,2025-26,9
